# Upload PropertyLens artifacts to Hugging Face

Pushes everything that lives outside git (large model bundles, XAI caches, feature tables) to two HF repos under [`PropertyLens`](https://huggingface.co/PropertyLens):

| Repo | Type | What goes there |
|---|---|---|
| `PropertyLens/final-propertylens-models` | model | `data/artifacts/**` + photo condition `.pth` |
| `PropertyLens/final-dataset` | dataset | `data/feature_data/**` + raw schooling extract |

The companion download notebooks under `notebooks/00_download_*.ipynb` pull these back into the same paths, so the backend Just Works after a fresh clone.

**Prereqs**
- `HF_TOKEN` (write-scoped) in `.env` or environment
- `huggingface_hub` installed (`pip install -q huggingface_hub python-dotenv`)
- Member of the `PropertyLens` org (or a fork you own)

Run cells top-to-bottom. The **Preview** cell is read-only — review the file list before running the upload cells.

> **First-run note:** Section 4 and 5 use `create_repo(..., exist_ok=True)`, so the new repos `final-propertylens-models` and `final-dataset` get auto-created the first time you upload.

## 1 — Setup

In [6]:
%pip install -q huggingface_hub python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import HfApi, login

REPO_ROOT = Path.cwd().resolve()
load_dotenv(REPO_ROOT / ".env")

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN missing. Add it to .env (HF_TOKEN=hf_...) or export it before running this cell."
    )

api = HfApi(token=HF_TOKEN)
whoami = api.whoami()
print(f"Logged in as: {whoami.get('name')} ({whoami.get('email', 'no-email')})")
print(f"Repo root: {REPO_ROOT}")

Logged in as: Bhuvesh09 (no-email)
Repo root: /Users/bhuvesh/Documents/PropertyLens


## 2 — Repo + path config

Each entry maps a **local folder** → **path inside the HF repo**, with optional include/ignore patterns. The download notebooks reverse this mapping.

In [8]:
MODEL_REPO   = "PropertyLens/final-propertylens-models"   # type: model
DATASET_REPO = "PropertyLens/final-dataset"               # type: dataset

# Common junk to skip in every upload.
COMMON_IGNORE = ["*.bak", "__pycache__/*", ".DS_Store", ".ipynb_checkpoints/*"]

MODEL_UPLOADS = [
    {
        "label": "Hybrid ensemble + XAI artifacts",
        "local":      REPO_ROOT / "data" / "artifacts",
        "path_in_repo": "artifacts",
        "allow": None,
        "ignore": COMMON_IGNORE,
    },
    {
        "label": "Photo condition model (EfficientNet-B0)",
        "local":      REPO_ROOT / "notebooks" / "05_photo_layer" / "artifacts",
        "path_in_repo": "05_photo_layer/artifacts",
        "allow": ["condition_model_*.pth", "condition_model_*_meta.json"],
        "ignore": COMMON_IGNORE,
    },
]

DATASET_UPLOADS = [
    {
        "label": "Feature tables (train / test / full)",
        "local":      REPO_ROOT / "data" / "feature_data",
        "path_in_repo": "feature_data",
        "allow": None,
        "ignore": COMMON_IGNORE + [".cache/*", "**/.cache/*"],
    },
    {
        "label": "Raw schooling extract",
        "local":      REPO_ROOT / "data" / "amenities",
        "path_in_repo": "amenities",
        "allow": ["sgschooling_*.csv"],
        "ignore": COMMON_IGNORE,
    },
]

for upl in MODEL_UPLOADS + DATASET_UPLOADS:
    if not upl["local"].exists():
        print(f"⚠️  Local folder missing — will be skipped: {upl['local']}")

## 3 — Preview (read-only)

Lists every file that would be uploaded, with sizes. **Nothing is uploaded yet.** Review the totals before running the upload cells below.

In [9]:
import fnmatch


def _matches_any(path: Path, patterns):
    rel = str(path)
    return any(fnmatch.fnmatch(path.name, p) or fnmatch.fnmatch(rel, p) for p in patterns)


def preview_upload(upl):
    local: Path = upl["local"]
    if not local.exists():
        print(f"  (folder not found — skipped: {local})")
        return 0, 0
    files = []
    for p in local.rglob("*"):
        if not p.is_file():
            continue
        rel = p.relative_to(local)
        if upl["allow"] and not any(fnmatch.fnmatch(p.name, pat) for pat in upl["allow"]):
            continue
        if upl["ignore"] and _matches_any(rel, upl["ignore"]):
            continue
        files.append(p)
    total_bytes = sum(p.stat().st_size for p in files)
    print(f"  {len(files)} file(s), {total_bytes / 1e6:,.1f} MB total")
    for p in sorted(files)[:30]:
        size_mb = p.stat().st_size / 1e6
        print(f"    {p.relative_to(local)}  ({size_mb:,.2f} MB)")
    if len(files) > 30:
        print(f"    … (+{len(files) - 30} more)")
    return len(files), total_bytes


print(f"=== {MODEL_REPO} (type: model) ===")
model_files = model_bytes = 0
for upl in MODEL_UPLOADS:
    print(f"\n[{upl['label']}] {upl['local']} → {upl['path_in_repo']}/")
    n, b = preview_upload(upl)
    model_files += n
    model_bytes += b

print(f"\n=== {DATASET_REPO} (type: dataset) ===")
ds_files = ds_bytes = 0
for upl in DATASET_UPLOADS:
    print(f"\n[{upl['label']}] {upl['local']} → {upl['path_in_repo']}/")
    n, b = preview_upload(upl)
    ds_files += n
    ds_bytes += b

print("\n" + "=" * 60)
print(f"Model repo total:   {model_files:>4} files, {model_bytes / 1e6:>8,.1f} MB")
print(f"Dataset repo total: {ds_files:>4} files, {ds_bytes / 1e6:>8,.1f} MB")
print(f"Combined:           {model_files + ds_files:>4} files, {(model_bytes + ds_bytes) / 1e6:>8,.1f} MB")

=== PropertyLens/final-propertylens-models (type: model) ===

[Hybrid ensemble + XAI artifacts] /Users/bhuvesh/Documents/PropertyLens/data/artifacts → artifacts/
  18 file(s), 685.9 MB total
    baseline_model_metrics_test.csv  (0.00 MB)
    hybrid_cluster_bundle.joblib  (271.02 MB)
    hybrid_cluster_feature_columns.json  (0.00 MB)
    hybrid_cluster_kmeans.joblib  (0.82 MB)
    hybrid_cluster_meta.json  (0.00 MB)
    hybrid_cluster_models.joblib  (138.77 MB)
    hybrid_xai/cbr_features.json  (0.00 MB)
    hybrid_xai/cbr_index.joblib  (13.78 MB)
    hybrid_xai/cbr_scaler.joblib  (0.00 MB)
    hybrid_xai/cbr_training_data.parquet  (3.67 MB)
    hybrid_xai/cluster_profiles.json  (0.00 MB)
    hybrid_xai/composite_treeshap_global_importance.json  (0.02 MB)
    hybrid_xai/global_shap_cache.json  (0.00 MB)
    hybrid_xai/hybrid_xai_meta.json  (0.00 MB)
    hybrid_xai/lime_training_data.joblib  (6.16 MB)
    hybrid_xai/rules.json  (0.03 MB)
    hybrid_xai/shap_explainers.joblib  (251.65 MB)

## 4 — Upload model artifacts → `PropertyLens/propertylens-models`

Creates the repo if missing. Each upload runs as a single commit; large files (>10 MB) are auto-LFS'd by `huggingface_hub`.

In [10]:
api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True, private=False)

for upl in MODEL_UPLOADS:
    if not upl["local"].exists():
        print(f"⏭️  Skipped (missing): {upl['local']}")
        continue
    print(f"⬆️  Uploading [{upl['label']}] {upl['local']} → {upl['path_in_repo']}/")
    api.upload_folder(
        repo_id=MODEL_REPO,
        repo_type="model",
        folder_path=str(upl["local"]),
        path_in_repo=upl["path_in_repo"],
        allow_patterns=upl["allow"],
        ignore_patterns=upl["ignore"],
        commit_message=f"Upload {upl['label']}",
    )
    print("   done.")

print("\n✅ Model repo upload complete.")
print(f"   https://huggingface.co/{MODEL_REPO}")

⬆️  Uploading [Hybrid ensemble + XAI artifacts] /Users/bhuvesh/Documents/PropertyLens/data/artifacts → artifacts/


Processing Files (9 / 9): 100%|██████████|  686MB /  686MB, 48.8MB/s  
New Data Upload: 100%|██████████| 2.90MB / 2.90MB,  558kB/s  


   done.
⬆️  Uploading [Photo condition model (EfficientNet-B0)] /Users/bhuvesh/Documents/PropertyLens/notebooks/05_photo_layer/artifacts → 05_photo_layer/artifacts/


Processing Files (1 / 1): 100%|██████████| 16.3MB / 16.3MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


   done.

✅ Model repo upload complete.
   https://huggingface.co/PropertyLens/final-propertylens-models


## 5 — Upload feature data → `PropertyLens/Resealeflats`

In [11]:
api.create_repo(repo_id=DATASET_REPO, repo_type="dataset", exist_ok=True, private=False)

for upl in DATASET_UPLOADS:
    if not upl["local"].exists():
        print(f"⏭️  Skipped (missing): {upl['local']}")
        continue
    print(f"⬆️  Uploading [{upl['label']}] {upl['local']} → {upl['path_in_repo']}/")
    api.upload_folder(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        folder_path=str(upl["local"]),
        path_in_repo=upl["path_in_repo"],
        allow_patterns=upl["allow"],
        ignore_patterns=upl["ignore"],
        commit_message=f"Upload {upl['label']}",
    )
    print("   done.")

print("\n✅ Dataset repo upload complete.")
print(f"   https://huggingface.co/datasets/{DATASET_REPO}")

⬆️  Uploading [Feature tables (train / test / full)] /Users/bhuvesh/Documents/PropertyLens/data/feature_data → feature_data/


Processing Files (9 / 9): 100%|██████████|  908MB /  908MB,  114MB/s  
New Data Upload: 100%|██████████| 98.8MB / 98.8MB, 12.3MB/s  


   done.
⬆️  Uploading [Raw schooling extract] /Users/bhuvesh/Documents/PropertyLens/data/amenities → amenities/
   done.

✅ Dataset repo upload complete.
   https://huggingface.co/datasets/PropertyLens/final-dataset


## 6 — Verify (read back the file lists from HF)

In [12]:
for repo_id, repo_type in [(MODEL_REPO, "model"), (DATASET_REPO, "dataset")]:
    print(f"=== {repo_id} ({repo_type}) ===")
    try:
        files = api.list_repo_files(repo_id=repo_id, repo_type=repo_type)
    except Exception as e:
        print(f"  (failed to list: {e})\n")
        continue
    for f in sorted(files):
        print(f"  {f}")
    print(f"  total: {len(files)} files\n")

=== PropertyLens/final-propertylens-models (model) ===
  .gitattributes
  05_photo_layer/artifacts/condition_model_20260413.pth
  artifacts/baseline_model_metrics_test.csv
  artifacts/hybrid_cluster_bundle.joblib
  artifacts/hybrid_cluster_feature_columns.json
  artifacts/hybrid_cluster_kmeans.joblib
  artifacts/hybrid_cluster_meta.json
  artifacts/hybrid_cluster_models.joblib
  artifacts/hybrid_xai/cbr_features.json
  artifacts/hybrid_xai/cbr_index.joblib
  artifacts/hybrid_xai/cbr_scaler.joblib
  artifacts/hybrid_xai/cbr_training_data.parquet
  artifacts/hybrid_xai/cluster_profiles.json
  artifacts/hybrid_xai/composite_treeshap_global_importance.json
  artifacts/hybrid_xai/global_shap_cache.json
  artifacts/hybrid_xai/hybrid_xai_meta.json
  artifacts/hybrid_xai/lime_training_data.joblib
  artifacts/hybrid_xai/rules.json
  artifacts/hybrid_xai/shap_explainers.joblib
  artifacts/hybrid_xai/surrogate_model.joblib
  total: 20 files

=== PropertyLens/final-dataset (dataset) ===
  .gitattr